# GenX simulations analyser

### Imports

In [1]:
import pandas as pd
import os
from xlsxwriter.color import Color

### Functions

In [2]:
def TES_cost(energy_capacity_MWht, power_capacity_MWt, reference_BoP_efficiency = .4, reference_TES_energy_cost_kWhe = 45, reference_TES_power_cost_kWe = 1645):
    """Calculate the total TES cost based on energy and power capacities."""
    energy_cost = (reference_TES_energy_cost_kWhe*1000*reference_BoP_efficiency) * energy_capacity_MWht
    power_cost = (reference_TES_power_cost_kWe*1000*reference_BoP_efficiency) * power_capacity_MWt
    return energy_cost + power_cost

def annualized_cost(cost, lifetime = 30, discount_rate = .06):
    """Calculate the annualized cost given total cost, lifetime in years, and discount rate."""
    if discount_rate == 0:
        return cost / lifetime
    annuity_factor = discount_rate / (1 - (1 + discount_rate) ** (-lifetime))
    return cost * annuity_factor

def scenario_parameters(scenario_path):
    """Returns the parameters of the scenario from its path"""
    if 'Scenario_00' in scenario_path:
        if 'No_TES_Cost_3000.0_EmissLevel_4.0_gCO2perkWh' in os.listdir(scenario_path):
            scenario_name = "BASELINE_Scenario"
    else:
        split = scenario_path.split("_")        
        scenario_name = f"Scenario_{split[-1][:-9]}"
    # end if
    return scenario_name

### Parameters

In [3]:
directory = './Scenarios/'
relevant_output_files = ['fusion_summary.csv', 'fusion_time_13.csv', 'costs.csv', 'NetRevenue.csv']

reference_BoP_efficiency = .4 # cf. Fusion study
reference_TES_energy_cost_kWhe = 45  # $/kWh_e cf. Fusion study
reference_TES_discharge_cost_kWe = 0  # $/kW_e cf. assumption "in-line"
reference_BoP_cost_kWe = 0  # $/kW_e cf. assumption "in-line"

reference_TES_power_cost_kWe = reference_TES_discharge_cost_kWe + reference_BoP_cost_kWe

reference_TES_economic_lifetime = 40  # years
reference_discount_rate = 0.06  # cf. Fusion study

### Import scenarios list

In [4]:
scenarios_list = []

with open('..\segment_scenarios_generation\Scenario_list.txt', 'r') as file:
    lines = file.readlines()

    for line in lines:
        line_scenario = []
        for y in line.split():
            line_scenario.append(int(y))
        # line_scenario.append(int(y) for y in line.split())
        # print(line_scenario)
        scenarios_list.append(line_scenario)
    # end for

### Initialisation of dictionaries

In [5]:
scenario_names_list = []
scenario_years_dict = {}
scenario_yearcount_dict = {}
otheroptions_list = []

# Initialize dictionaries to store absolute data
a_Reactor_Thermal_Capacity_dict = {}
a_Turbine_Gross_Capacity_dict = {}
a_Turbine_Net_Capacity_dict = {}
a_Num_Reactors_dict = {}

a_TES_Energy_Capacity_dict = {}
a_TES_Discharge_Capacity_dict = {}
a_TES_Charge_Capacity_dict = {}
a_TES_relDisCha_Capacity_dict = {}
a_TES_duration_on = {}
a_TES_duration = {}
a_TES_energy_ccost_ass = {}
a_TES_on_cost = {}
a_TES_ann_cost = {}
a_TES_total_ann_cost = {}

a_cTotal_dict = {}
a_cFix_dict = {}
a_cVar_dict = {}
a_cNSE_dict = {}
a_cStart_dict = {}

a_FPP_Total_revenue_dict = {}
a_FPP_Energy_revenue_dict = {}
a_FPP_Subsidies_revenue_dict = {}
a_FPP_Total_cost_dict = {}
a_FPP_InvestmentMW_cost_dict = {}
a_FPP_InvestmentMWh_cost_dict = {}
a_FPP_FOMMW_cost_dict = {}
a_FPP_FOMMWh_cost_dict = {}
a_FPP_VOMout_cost_dict = {}
a_FPP_Fuel_cost_dict = {}
a_FPP_VOMin_cost_dict = {}
a_FPP_Start_cost_dict = {}
a_FPP_Charge_cost_dict = {}
a_FPP_Emissions_cost_dict = {}

a_FPP_net_revenue_dict = {}

# # Initialize dictionaries to store delta data (relative to baseline)
# d_Reactor_Thermal_Capacity_dict = {}
# d_Turbine_Gross_Capacity_dict = {}
# d_Turbine_Net_Capacity_dict = {}
# d_Num_Reactors_dict = {}

# d_TES_Energy_Capacity_dict = {}
# d_TES_Discharge_Capacity_dict = {}
# d_TES_duration = {}
# d_TES_on_cost = {}
# d_TES_ann_cost = {}
# d_TES_total_ann_cost = {}

# d_cTotal_dict = {}
# d_cFix_dict = {}
# d_cVar_dict = {}
# d_cNSE_dict = {}
# d_cStart_dict = {}

# d_FPP_Total_revenue_dict = {}
# d_FPP_Energy_revenue_dict = {}
# d_FPP_Subsidies_revenue_dict = {}
# d_FPP_Total_cost_dict = {}
# d_FPP_InvestmentMW_cost_dict = {}
# d_FPP_InvestmentMWh_cost_dict = {}
# d_FPP_FOMMW_cost_dict = {}
# d_FPP_FOMMWh_cost_dict = {}
# d_FPP_VOMout_cost_dict = {}
# d_FPP_Fuel_cost_dict = {}
# d_FPP_VOMin_cost_dict = {}
# d_FPP_Start_cost_dict = {}
# d_FPP_Charge_cost_dict = {}
# d_FPP_Emissions_cost_dict = {}

# d_FPP_net_revenue_dict = {}

# # Initialize dictionaries to store relative data (delta as a fraction of baseline value)
# r_Reactor_Thermal_Capacity_dict = {}
# r_Turbine_Gross_Capacity_dict = {}
# r_Turbine_Net_Capacity_dict = {}
# r_Num_Reactors_dict = {}

# r_TES_Energy_Capacity_dict = {}
# r_TES_Discharge_Capacity_dict = {}
# r_TES_duration = {}
# r_TES_on_cost = {}
# r_TES_ann_cost = {}
# r_TES_total_ann_cost = {}

# r_cTotal_dict = {}
# r_cFix_dict = {}
# r_cVar_dict = {}
# r_cNSE_dict = {}
# r_cStart_dict = {}

# r_FPP_Total_revenue_dict = {}
# r_FPP_Energy_revenue_dict = {}
# r_FPP_Subsidies_revenue_dict = {}
# r_FPP_Total_cost_dict = {}
# r_FPP_InvestmentMW_cost_dict = {}
# r_FPP_InvestmentMWh_cost_dict = {}
# r_FPP_FOMMW_cost_dict = {}
# r_FPP_FOMMWh_cost_dict = {}
# r_FPP_VOMout_cost_dict = {}
# r_FPP_Fuel_cost_dict = {}
# r_FPP_VOMin_cost_dict = {}
# r_FPP_Start_cost_dict = {}
# r_FPP_Charge_cost_dict = {}
# r_FPP_Emissions_cost_dict = {}

# r_FPP_net_revenue_dict = {}

empty_df_dict = {}
# FPP_cost_dict = {}
# FPP_cCapacity_dict = {}
# Carbon_constraint_dict = {}
# TES_Duration_Capacity_cdict  = {}
# TES_Discharge_Cap_cdict  = {}

TES_max_charge_rate_dict = {}
TES_max_discharge_rate_dict = {}
TES_min_net_discharge_rate_dict = {}
TES_max_net_discharge_rate_dict = {}
TES_suspected_attractor_dict = {}

### Scenario analysis

In [6]:
for scenario in os.listdir(directory):
    # print(scenario)
    scenario_path = directory + scenario + '/Results/'
    scenario_name = scenario_parameters(scenario_path)
    if scenario_name == "BASELINE_Scenario":
        run_path = os.path.join(scenario_path, 'No_TES_Cost_3000.0_EmissLevel_4.0_gCO2perkWh/')
    else:
        run_path = os.path.join(scenario_path, 'OK_Cost_3000.0_EmissLevel_4.0_gCO2perkWh/')
    # print(run_path)
    if os.path.isdir(run_path):
        print(f"Processing: {scenario_name}")
        scenario_id = scenario.split('_')[-1]
        scenario_years_dict.update({scenario_name : str(scenarios_list[int(scenario_id)])[1:-1]})
        scenario_yearcount_dict.update({scenario_name : len(scenarios_list[int(scenario_id)])})

        cost_df = pd.read_csv(directory + scenario + '/Fusion_data.csv', index_col=0)

        empty_df_dict.update({scenario_name : ""})
        a_TES_energy_ccost_ass.update({scenario_name : float(cost_df['Stor_Cost_per_MWht'].loc['fusion_2'])})
        
        scenario_names_list.append(scenario_name)

        for file in os.listdir(run_path+"/fusion/") + os.listdir(run_path):
            # print(f"  Checking file: {file} from {scenario_name}")
            if file in relevant_output_files:
                # print(f"    Processing file: {file}")
                if 'fusion' in file:
                    file_path = os.path.join(run_path, 'fusion/', file)
                else:
                    file_path = os.path.join(run_path, file)
                # end if

                if file_path.endswith('fusion_summary.csv'):
                    # Process fusion_summary.csv
                    df = pd.read_csv(file_path)

                    a_Reactor_Thermal_Capacity_dict.update({scenario_name : float(df['Reactor Thermal Capacity MWt'].iloc[0])})
                    a_Turbine_Gross_Capacity_dict.update({scenario_name : float(df['Turbine Gross Capacity MWe'].iloc[0])})
                    a_Turbine_Net_Capacity_dict.update({scenario_name : float(df['Turbine Net Capacity MWe'].iloc[0])})
                    a_Num_Reactors_dict.update({scenario_name : float(df['Num Reactors'].iloc[0])})

                    a_TES_Energy_Capacity_dict.update({scenario_name : float(df['Thermal Storage Energy Capacity MWht'].iloc[0])})
                    a_TES_Discharge_Capacity_dict.update({scenario_name : float(df['Thermal Storage Discharge Capacity MWt'].iloc[0])})
                    a_TES_Charge_Capacity_dict.update({scenario_name : float(df['Thermal Storage Charge Capacity MWt'].iloc[0])})
                    
                    if scenario_name == "BASELINE_Scenario":
                        a_TES_relDisCha_Capacity_dict.update({scenario_name : "Undefined"})
                        a_TES_duration_on.update({scenario_name : "Undefined"})
                        a_TES_duration.update({scenario_name : "Undefined"})
                    else:
                        a_TES_relDisCha_Capacity_dict.update({scenario_name : float(df['Thermal Storage Discharge Capacity MWt'].iloc[0] / df['Thermal Storage Charge Capacity MWt'].iloc[0])})
                        a_TES_duration_on.update({scenario_name : float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) / (float(df['Thermal Storage Discharge Capacity MWt'].iloc[0]) - float(df['Thermal Storage Charge Capacity MWt'].iloc[0]))})
                        a_TES_duration.update({scenario_name : float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) / float(df['Thermal Storage Discharge Capacity MWt'].iloc[0])})
                    # end if
                    a_TES_on_cost.update({scenario_name : TES_cost(float(df['Thermal Storage Energy Capacity MWht'].iloc[0]), float(df['Thermal Storage Discharge Capacity MWt'].iloc[0]), reference_BoP_efficiency, reference_TES_energy_cost_kWhe, reference_TES_power_cost_kWe)})
                    a_TES_ann_cost.update({scenario_name : annualized_cost(a_TES_on_cost[scenario_name], reference_TES_economic_lifetime, reference_discount_rate)})
                    a_TES_total_ann_cost.update({scenario_name : a_TES_ann_cost[scenario_name]*reference_TES_economic_lifetime})

                    # if scenario_name.startswith('BASELINE_'):
                    #     # Save baseline values
                    #     baseline_Reactor_Thermal_Capacity = float(df['Reactor Thermal Capacity MWt'].iloc[0])
                    #     baseline_Turbine_Gross_Capacity = float(df['Turbine Gross Capacity MWe'].iloc[0])
                    #     baseline_Turbine_Net_Capacity = float(df['Turbine Net Capacity MWe'].iloc[0])
                    #     baseline_Num_Reactors = float(df['Num Reactors'].iloc[0])

                    #     baseline_TES_Energy_Capacity = float(df['Thermal Storage Energy Capacity MWht'].iloc[0])
                    #     baseline_TES_Discharge_Capacity = float(df['Thermal Storage Discharge Capacity MWt'].iloc[0])
                    #     baseline_TES_duration = float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) / float(df['Thermal Storage Discharge Capacity MWt'].iloc[0])

                    #     baseline_TES_on_cost = TES_cost(baseline_TES_Energy_Capacity, baseline_TES_Discharge_Capacity, reference_BoP_efficiency, reference_TES_energy_cost_kWhe, reference_TES_power_cost_kWe)
                    #     baseline_TES_ann_cost = annualized_cost(baseline_TES_on_cost, reference_TES_economic_lifetime, reference_discount_rate)
                    #     baseline_TES_total_ann_cost = baseline_TES_ann_cost * reference_TES_economic_lifetime
                    # # end if

                    # d_Reactor_Thermal_Capacity_dict.update({scenario_name : float(df['Reactor Thermal Capacity MWt'].iloc[0]) - baseline_Reactor_Thermal_Capacity})
                    # d_Turbine_Gross_Capacity_dict.update({scenario_name : float(df['Turbine Gross Capacity MWe'].iloc[0]) - baseline_Turbine_Gross_Capacity})
                    # d_Turbine_Net_Capacity_dict.update({scenario_name : float(df['Turbine Net Capacity MWe'].iloc[0]) - baseline_Turbine_Net_Capacity})
                    # d_Num_Reactors_dict.update({scenario_name : float(df['Num Reactors'].iloc[0]) - baseline_Num_Reactors})

                    # d_TES_Energy_Capacity_dict.update({scenario_name : float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) - baseline_TES_Energy_Capacity})
                    # d_TES_Discharge_Capacity_dict.update({scenario_name : float(df['Thermal Storage Discharge Capacity MWt'].iloc[0]) - baseline_TES_Discharge_Capacity})
                    # d_TES_duration.update({scenario_name : (float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) / float(df['Thermal Storage Discharge Capacity MWt'].iloc[0])) - baseline_TES_duration})
                    # d_TES_on_cost.update({scenario_name : TES_cost(float(df['Thermal Storage Energy Capacity MWht'].iloc[0]), float(df['Thermal Storage Discharge Capacity MWt'].iloc[0]), reference_BoP_efficiency, reference_TES_energy_cost_kWhe, reference_TES_power_cost_kWe) - baseline_TES_on_cost})
                    # d_TES_ann_cost.update({scenario_name : annualized_cost(a_TES_on_cost[scenario_name], reference_TES_economic_lifetime, reference_discount_rate) - baseline_TES_ann_cost})
                    # d_TES_total_ann_cost.update({scenario_name : a_TES_ann_cost[scenario_name]*reference_TES_economic_lifetime - baseline_TES_total_ann_cost})

                    # r_Reactor_Thermal_Capacity_dict.update({scenario_name : (float(df['Reactor Thermal Capacity MWt'].iloc[0]) - baseline_Reactor_Thermal_Capacity) / baseline_Reactor_Thermal_Capacity if baseline_Reactor_Thermal_Capacity != 0 else None})
                    # r_Turbine_Gross_Capacity_dict.update({scenario_name : (float(df['Turbine Gross Capacity MWe'].iloc[0]) - baseline_Turbine_Gross_Capacity) / baseline_Turbine_Gross_Capacity if baseline_Turbine_Gross_Capacity != 0 else None})
                    # r_Turbine_Net_Capacity_dict.update({scenario_name : (float(df['Turbine Net Capacity MWe'].iloc[0]) - baseline_Turbine_Net_Capacity) / baseline_Turbine_Net_Capacity if baseline_Turbine_Net_Capacity != 0 else None})
                    # r_Num_Reactors_dict.update({scenario_name : (float(df['Num Reactors'].iloc[0]) - baseline_Num_Reactors) / baseline_Num_Reactors if baseline_Num_Reactors != 0 else None})

                    # r_TES_Energy_Capacity_dict.update({scenario_name : (float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) - baseline_TES_Energy_Capacity) / baseline_TES_Energy_Capacity if baseline_TES_Energy_Capacity != 0 else None})
                    # r_TES_Discharge_Capacity_dict.update({scenario_name : (float(df['Thermal Storage Discharge Capacity MWt'].iloc[0]) - baseline_TES_Discharge_Capacity) / baseline_TES_Discharge_Capacity if baseline_TES_Discharge_Capacity != 0 else None})
                    # r_TES_duration.update({scenario_name : ((float(df['Thermal Storage Energy Capacity MWht'].iloc[0]) / float(df['Thermal Storage Discharge Capacity MWt'].iloc[0])) - baseline_TES_duration) / baseline_TES_duration if baseline_TES_duration != 0 else None})
                    # r_TES_on_cost.update({scenario_name : (TES_cost(float(df['Thermal Storage Energy Capacity MWht'].iloc[0]), float(df['Thermal Storage Discharge Capacity MWt'].iloc[0]), reference_BoP_efficiency, reference_TES_energy_cost_kWhe, reference_TES_power_cost_kWe) - baseline_TES_on_cost) / baseline_TES_on_cost if baseline_TES_on_cost != 0 else None})
                    # r_TES_ann_cost.update({scenario_name : (annualized_cost(a_TES_on_cost[scenario_name], reference_TES_economic_lifetime, reference_discount_rate) - baseline_TES_ann_cost) / baseline_TES_ann_cost if baseline_TES_ann_cost != 0 else None})
                    # r_TES_total_ann_cost.update({scenario_name : (a_TES_ann_cost[scenario_name]*reference_TES_economic_lifetime - baseline_TES_total_ann_cost) / baseline_TES_total_ann_cost if baseline_TES_total_ann_cost != 0 else None})
                    continue
                # end if

                if file_path.endswith('fusion_time_13.csv'):
                    # Process fusion_time_13.csv
                    df = pd.read_csv(file_path)

                    if scenario_name == "BASELINE_Scenario":
                        TES_suspected_attractor = "Undefined"
                    
                        TES_max_charge_rate_dict.update({scenario_name : "Undefined"})
                        TES_max_discharge_rate_dict.update({scenario_name : "Undefined"})
                        TES_min_net_discharge_rate_dict.update({scenario_name : "Undefined"})
                        TES_max_net_discharge_rate_dict.update({scenario_name : "Undefined"})
                        TES_suspected_attractor_dict.update({scenario_name : TES_suspected_attractor})
                    else:
                        TES_suspected_attractor = float(df['Thermal Storage Inventory MWht'][abs(df['Thermal Storage Net Discharge MWht']) < .01 * a_TES_Discharge_Capacity_dict[scenario_name]].mean())  / a_TES_Energy_Capacity_dict[scenario_name]
                    
                        TES_max_charge_rate_dict.update({scenario_name : float(df['Thermal Storage Charging MWht'].max()) / a_TES_Charge_Capacity_dict[scenario_name]})
                        TES_max_discharge_rate_dict.update({scenario_name : float(df['Thermal Storage Discharging MWht'].max()) / a_TES_Discharge_Capacity_dict[scenario_name]})
                        TES_min_net_discharge_rate_dict.update({scenario_name : -float(df['Thermal Storage Net Discharge MWht'].min()) / a_TES_Charge_Capacity_dict[scenario_name]})
                        TES_max_net_discharge_rate_dict.update({scenario_name : float(df['Thermal Storage Net Discharge MWht'].max()) / a_TES_Discharge_Capacity_dict[scenario_name]})
                        TES_suspected_attractor_dict.update({scenario_name : TES_suspected_attractor})
                    # end if
                # end if

                if file_path.endswith('costs.csv'):
                    # Process costs.csv

                    df = pd.read_csv(file_path)

                    first_row = 0

                    a_cTotal_dict.update({scenario_name : [float(df['Total'][first_row])]})
                    a_cFix_dict.update({scenario_name : [float(df['Total'][first_row + 1])]})
                    a_cVar_dict.update({scenario_name : [float(df['Total'][first_row + 2])]})
                    a_cNSE_dict.update({scenario_name : [float(df['Total'][first_row + 3])]})
                    a_cStart_dict.update({scenario_name : [float(df['Total'][first_row + 4])]})
                
                    # if scenario_name.startswith('BASELINE_'):
                    #     # Save baseline costs
                    #     baseline_cTotal = float(df['Total'][first_row])
                    #     baseline_cFix = float(df['Total'][first_row + 1])
                    #     baseline_cVar = float(df['Total'][first_row + 2])
                    #     baseline_cNSE = float(df['Total'][first_row + 3])
                    #     baseline_cStart = float(df['Total'][first_row + 4])
                    # # end if

                    # d_cTotal_dict.update({scenario_name : [float(df['Total'][first_row]) - baseline_cTotal]})
                    # d_cFix_dict.update({scenario_name : [float(df['Total'][first_row + 1]) - baseline_cFix]})
                    # d_cVar_dict.update({scenario_name : [float(df['Total'][first_row + 2]) - baseline_cVar]})
                    # d_cNSE_dict.update({scenario_name : [float(df['Total'][first_row + 3]) - baseline_cNSE]})
                    # d_cStart_dict.update({scenario_name : [float(df['Total'][first_row + 4]) - baseline_cStart]})

                    # r_cTotal_dict.update({scenario_name : [(float(df['Total'][first_row]) - baseline_cTotal) / baseline_cTotal if baseline_cTotal != 0 else None]})
                    # r_cFix_dict.update({scenario_name : [(float(df['Total'][first_row + 1]) - baseline_cFix) / baseline_cFix if baseline_cFix != 0 else None]})
                    # r_cVar_dict.update({scenario_name : [(float(df['Total'][first_row + 2]) - baseline_cVar) / baseline_cVar if baseline_cVar != 0 else None]})
                    # r_cNSE_dict.update({scenario_name : [(float(df['Total'][first_row + 3]) - baseline_cNSE) / baseline_cNSE if baseline_cNSE != 0 else None]})
                    # r_cStart_dict.update({scenario_name : [(float(df['Total'][first_row + 4]) - baseline_cStart) / baseline_cStart if baseline_cStart != 0 else None]})

                    continue
                # end if

                if file_path.endswith('NetRevenue.csv'):
                    # Process NetRevenue.csv

                    df = pd.read_csv(file_path)

                    FPP_idx = 12
                    if df['Resource'][FPP_idx] != 'fusion_2':
                        print(f"Warning: Expected 'fusion_2' at index {FPP_idx}, found '{df['Resource'][FPP_idx]}' instead.")
                    # end if

                    a_FPP_Total_revenue_dict.update({scenario_name : [float(df['Revenue'][FPP_idx])]})
                    a_FPP_Energy_revenue_dict.update({scenario_name : [float(df['EnergyRevenue'][FPP_idx])]})
                    a_FPP_Subsidies_revenue_dict.update({scenario_name : [float(df['SubsidyRevenue'][FPP_idx])]})

                    a_FPP_Total_cost_dict.update({scenario_name : [float(df['Cost'][FPP_idx])]})
                    a_FPP_InvestmentMW_cost_dict.update({scenario_name : [float(df['Inv_cost_MW'][FPP_idx])]})
                    a_FPP_InvestmentMWh_cost_dict.update({scenario_name : [float(df['Inv_cost_MWh'][FPP_idx])]})
                    a_FPP_FOMMW_cost_dict.update({scenario_name : [float(df['Fixed_OM_cost_MW'][FPP_idx])]})
                    a_FPP_FOMMWh_cost_dict.update({scenario_name : [float(df['Fixed_OM_cost_MWh'][FPP_idx])]})
                    a_FPP_VOMout_cost_dict.update({scenario_name : [float(df['Var_OM_cost_out'][FPP_idx])]})
                    a_FPP_Fuel_cost_dict.update({scenario_name : [float(df['Fuel_cost'][FPP_idx])]})
                    a_FPP_VOMin_cost_dict.update({scenario_name : [float(df['Var_OM_cost_in'][FPP_idx])]})
                    a_FPP_Start_cost_dict.update({scenario_name : [float(df['StartCost'][FPP_idx])]})
                    a_FPP_Charge_cost_dict.update({scenario_name : [float(df['Charge_cost'][FPP_idx])]})
                    a_FPP_Emissions_cost_dict.update({scenario_name : [float(df['EmissionsCost'][FPP_idx])]})
                    
                    a_FPP_net_revenue_dict.update({scenario_name : [float(df['Revenue'][FPP_idx]) - float(df['Cost'][FPP_idx])]})
                
                    # if scenario_name.startswith('BASELINE_'):
                    #     # Save baseline revenues and costs
                    #     baseline_FPP_Total_revenue = float(df['Revenue'][FPP_idx])
                    #     baseline_FPP_Energy_revenue = float(df['EnergyRevenue'][FPP_idx])
                    #     baseline_FPP_Subsidies_revenue = float(df['SubsidyRevenue'][FPP_idx])

                    #     baseline_FPP_Total_cost = float(df['Cost'][FPP_idx])
                    #     baseline_FPP_InvestmentMW_cost = float(df['Inv_cost_MW'][FPP_idx])
                    #     baseline_FPP_InvestmentMWh_cost = float(df['Inv_cost_MWh'][FPP_idx])
                    #     baseline_FPP_FOMMW_cost = float(df['Fixed_OM_cost_MW'][FPP_idx])
                    #     baseline_FPP_FOMMWh_cost = float(df['Fixed_OM_cost_MWh'][FPP_idx])
                    #     baseline_FPP_VOMout_cost = float(df['Var_OM_cost_out'][FPP_idx])
                    #     baseline_FPP_Fuel_cost = float(df['Fuel_cost'][FPP_idx])
                    #     baseline_FPP_VOMin_cost = float(df['Var_OM_cost_in'][FPP_idx])
                    #     baseline_FPP_Start_cost = float(df['StartCost'][FPP_idx])
                    #     baseline_FPP_Charge_cost = float(df['Charge_cost'][FPP_idx])
                    #     baseline_FPP_Emissions_cost = float(df['EmissionsCost'][FPP_idx])
                    # # end if

                    # d_FPP_Total_revenue_dict.update({scenario_name : [float(df['Revenue'][FPP_idx]) - baseline_FPP_Total_revenue]})
                    # d_FPP_Energy_revenue_dict.update({scenario_name : [float(df['EnergyRevenue'][FPP_idx]) - baseline_FPP_Energy_revenue]})
                    # d_FPP_Subsidies_revenue_dict.update({scenario_name : [float(df['SubsidyRevenue'][FPP_idx])  - baseline_FPP_Subsidies_revenue]})
                    
                    # d_FPP_Total_cost_dict.update({scenario_name : [float(df['Cost'][FPP_idx]) - baseline_FPP_Total_cost]})
                    # d_FPP_InvestmentMW_cost_dict.update({scenario_name : [float(df['Inv_cost_MW'][FPP_idx]) - baseline_FPP_InvestmentMW_cost]})
                    # d_FPP_InvestmentMWh_cost_dict.update({scenario_name : [float(df['Inv_cost_MWh'][FPP_idx]) - baseline_FPP_InvestmentMWh_cost]})
                    # d_FPP_FOMMW_cost_dict.update({scenario_name : [float(df['Fixed_OM_cost_MW'][FPP_idx]) - baseline_FPP_FOMMW_cost]})
                    # d_FPP_FOMMWh_cost_dict.update({scenario_name : [float(df['Fixed_OM_cost_MWh'][FPP_idx]) - baseline_FPP_FOMMWh_cost]})
                    # d_FPP_VOMout_cost_dict.update({scenario_name : [float(df['Var_OM_cost_out'][FPP_idx]) - baseline_FPP_VOMout_cost]})
                    # d_FPP_Fuel_cost_dict.update({scenario_name : [float(df['Fuel_cost'][FPP_idx]) - baseline_FPP_Fuel_cost]})
                    # d_FPP_VOMin_cost_dict.update({scenario_name : [float(df['Var_OM_cost_in'][FPP_idx]) - baseline_FPP_VOMin_cost]})
                    # d_FPP_Start_cost_dict.update({scenario_name : [float(df['StartCost'][FPP_idx]) - baseline_FPP_Start_cost]})
                    # d_FPP_Charge_cost_dict.update({scenario_name : [float(df['Charge_cost'][FPP_idx]) - baseline_FPP_Charge_cost]})
                    # d_FPP_Emissions_cost_dict.update({scenario_name : [float(df['EmissionsCost'][FPP_idx]) - baseline_FPP_Emissions_cost]})

                    # d_FPP_net_revenue_dict.update({scenario_name : [(float(df['Revenue'][FPP_idx]) - float(df['Cost'][FPP_idx])) - (baseline_FPP_Total_revenue - baseline_FPP_Total_cost)]})

                    # r_FPP_Total_revenue_dict.update({scenario_name : [(float(df['Revenue'][FPP_idx]) - baseline_FPP_Total_revenue) / baseline_FPP_Total_revenue if baseline_FPP_Total_revenue != 0 else None]})
                    # r_FPP_Energy_revenue_dict.update({scenario_name : [(float(df['EnergyRevenue'][FPP_idx]) - baseline_FPP_Energy_revenue) / baseline_FPP_Energy_revenue if baseline_FPP_Energy_revenue != 0 else None]})
                    # r_FPP_Subsidies_revenue_dict.update({scenario_name : [(float(df['SubsidyRevenue'][FPP_idx]) - baseline_FPP_Subsidies_revenue) / baseline_FPP_Subsidies_revenue if baseline_FPP_Subsidies_revenue != 0 else None]})
                    
                    # r_FPP_Total_cost_dict.update({scenario_name : [(float(df['Cost'][FPP_idx]) - baseline_FPP_Total_cost) / baseline_FPP_Total_cost if baseline_FPP_Total_cost != 0 else None]})
                    # r_FPP_InvestmentMW_cost_dict.update({scenario_name : [(float(df['Inv_cost_MW'][FPP_idx]) - baseline_FPP_InvestmentMW_cost) / baseline_FPP_InvestmentMW_cost if baseline_FPP_InvestmentMW_cost != 0 else None]})
                    # r_FPP_InvestmentMWh_cost_dict.update({scenario_name : [(float(df['Inv_cost_MWh'][FPP_idx]) - baseline_FPP_InvestmentMWh_cost) / baseline_FPP_InvestmentMWh_cost if baseline_FPP_InvestmentMWh_cost != 0 else None]})
                    # r_FPP_FOMMW_cost_dict.update({scenario_name : [(float(df['Fixed_OM_cost_MW'][FPP_idx]) - baseline_FPP_FOMMW_cost) / baseline_FPP_FOMMW_cost if baseline_FPP_FOMMW_cost != 0 else None]})
                    # r_FPP_FOMMWh_cost_dict.update({scenario_name : [(float(df['Fixed_OM_cost_MWh'][FPP_idx]) - baseline_FPP_FOMMWh_cost) / baseline_FPP_FOMMWh_cost if baseline_FPP_FOMMWh_cost != 0 else None]})
                    # r_FPP_VOMout_cost_dict.update({scenario_name : [(float(df['Var_OM_cost_out'][FPP_idx]) - baseline_FPP_VOMout_cost) / baseline_FPP_VOMout_cost if baseline_FPP_VOMout_cost != 0 else None]})
                    # r_FPP_Fuel_cost_dict.update({scenario_name : [(float(df['Fuel_cost'][FPP_idx]) - baseline_FPP_Fuel_cost) / baseline_FPP_Fuel_cost if baseline_FPP_Fuel_cost != 0 else None]})
                    # r_FPP_VOMin_cost_dict.update({scenario_name : [(float(df['Var_OM_cost_in'][FPP_idx]) - baseline_FPP_VOMin_cost) / baseline_FPP_VOMin_cost if baseline_FPP_VOMin_cost != 0 else None]})
                    # r_FPP_Start_cost_dict.update({scenario_name : [(float(df['StartCost'][FPP_idx]) - baseline_FPP_Start_cost) / baseline_FPP_Start_cost if baseline_FPP_Start_cost != 0 else None]})
                    # r_FPP_Charge_cost_dict.update({scenario_name : [(float(df['Charge_cost'][FPP_idx]) - baseline_FPP_Charge_cost) / baseline_FPP_Charge_cost if baseline_FPP_Charge_cost != 0 else None]})
                    # r_FPP_Emissions_cost_dict.update({scenario_name : [(float(df['EmissionsCost'][FPP_idx]) - baseline_FPP_Emissions_cost) / baseline_FPP_Emissions_cost if baseline_FPP_Emissions_cost != 0 else None]})

                    # r_FPP_net_revenue_dict.update({scenario_name : [((float(df['Revenue'][FPP_idx]) - float(df['Cost'][FPP_idx])) - (baseline_FPP_Total_revenue - baseline_FPP_Total_cost)) / (baseline_FPP_Total_revenue - baseline_FPP_Total_cost) if (baseline_FPP_Total_revenue - baseline_FPP_Total_cost) != 0 else None]})
                    # continue
                # end if
            # end if
        # end for
    # end if
# end for

Processing: BASELINE_Scenario
Processing: Scenario_01
Processing: Scenario_02
Processing: Scenario_03
Processing: Scenario_04
Processing: Scenario_05
Processing: Scenario_06
Processing: Scenario_07
Processing: Scenario_08
Processing: Scenario_09
Processing: Scenario_10
Processing: Scenario_11
Processing: Scenario_12
Processing: Scenario_13
Processing: Scenario_14
Processing: Scenario_15
Processing: Scenario_16
Processing: Scenario_17
Processing: Scenario_18
Processing: Scenario_19
Processing: Scenario_20
Processing: Scenario_21
Processing: Scenario_22
Processing: Scenario_23
Processing: Scenario_24
Processing: Scenario_25
Processing: Scenario_26
Processing: Scenario_27
Processing: Scenario_28
Processing: Scenario_29
Processing: Scenario_30
Processing: Scenario_31
Processing: Scenario_32
Processing: Scenario_33
Processing: Scenario_34
Processing: Scenario_35
Processing: Scenario_36
Processing: Scenario_37
Processing: Scenario_38
Processing: Scenario_39
Processing: Scenario_40
Processing

### DataFrames

#### Initialisation

In [7]:
a_scenario_years_df = pd.DataFrame(scenario_years_dict, index=["Years included"])
a_scenario_yearcount_df = pd.DataFrame(scenario_yearcount_dict, index=["Scenario length (years)"])
a_TES_energy_ccost_ass_df = pd.DataFrame(a_TES_energy_ccost_ass, index=["TES energy capital cost ($/MWh_t)"])

a_Reactor_Thermal_Capacity_df = pd.DataFrame(a_Reactor_Thermal_Capacity_dict, index=["Reactor Thermal Capacity MWt"])
a_Turbine_Gross_Capacity_df = pd.DataFrame(a_Turbine_Gross_Capacity_dict, index=["Turbine Gross Capacity MWe"])
a_Turbine_Net_Capacity_df = pd.DataFrame(a_Turbine_Net_Capacity_dict, index=["Turbine Net Capacity MWe"])
a_Num_Reactors_df = pd.DataFrame(a_Num_Reactors_dict, index=["Num Reactors"])

a_TES_Energy_Capacity_df = pd.DataFrame(a_TES_Energy_Capacity_dict, index=["Thermal Storage Energy Capacity MWht"])
a_TES_Discharge_Capacity_df = pd.DataFrame(a_TES_Discharge_Capacity_dict, index=["Thermal Storage Discharge Capacity MWt"])
a_TES_Charge_Capacity_df = pd.DataFrame(a_TES_Charge_Capacity_dict, index=["Thermal Storage Charge Capacity MWt"])
a_TES_relDisCha_Capacity_df = pd.DataFrame(a_TES_relDisCha_Capacity_dict, index=["Thermal Storage Boost ratio (Discharge/Charge)"])
a_TES_duration_on_df = pd.DataFrame(a_TES_duration_on, index=["Thermal Storage Duration hours [NPP baseload]"])
a_TES_duration_df = pd.DataFrame(a_TES_duration, index=["Thermal Storage Duration hours [NPP offline]"])
a_TES_on_cost_df = pd.DataFrame(a_TES_on_cost, index=["Thermal Storage Overnight Cost $"])
a_TES_ann_cost_df = pd.DataFrame(a_TES_ann_cost, index=["Thermal Storage Annualized Cost $/year"])
a_TES_total_ann_cost_df = pd.DataFrame(a_TES_total_ann_cost, index=["Thermal Storage Total Annualized Cost $"])

# d_Reactor_Thermal_Capacity_df = pd.DataFrame(d_Reactor_Thermal_Capacity_dict, index=["Reactor Thermal Capacity MWt"])
# d_Turbine_Gross_Capacity_df = pd.DataFrame(d_Turbine_Gross_Capacity_dict, index=["Turbine Gross Capacity MWe"])
# d_Turbine_Net_Capacity_df = pd.DataFrame(d_Turbine_Net_Capacity_dict, index=["Turbine Net Capacity MWe"])
# d_Num_Reactors_df = pd.DataFrame(d_Num_Reactors_dict, index=["Num Reactors"])

# d_TES_Energy_Capacity_df = pd.DataFrame(d_TES_Energy_Capacity_dict, index=["Thermal Storage Energy Capacity MWht"])
# d_TES_Discharge_Capacity_df = pd.DataFrame(d_TES_Discharge_Capacity_dict, index=["Thermal Storage Discharge Capacity MWt"])
# d_TES_duration_df = pd.DataFrame(d_TES_duration, index=["Thermal Storage Duration hours"])
# d_TES_on_cost_df = pd.DataFrame(d_TES_on_cost, index=["Thermal Storage Overnight Cost $"])
# d_TES_ann_cost_df = pd.DataFrame(d_TES_ann_cost, index=["Thermal Storage Annualized Cost $/year"])
# d_TES_total_ann_cost_df = pd.DataFrame(d_TES_total_ann_cost, index=["Thermal Storage Total Annualized Cost $"])

# r_Reactor_Thermal_Capacity_df = pd.DataFrame(r_Reactor_Thermal_Capacity_dict, index=["Reactor Thermal Capacity MWt"])
# r_Turbine_Gross_Capacity_df = pd.DataFrame(r_Turbine_Gross_Capacity_dict, index=["Turbine Gross Capacity MWe"])
# r_Turbine_Net_Capacity_df = pd.DataFrame(r_Turbine_Net_Capacity_dict, index=["Turbine Net Capacity MWe"])
# r_Num_Reactors_df = pd.DataFrame(r_Num_Reactors_dict, index=["Num Reactors"])

# r_TES_Energy_Capacity_df = pd.DataFrame(r_TES_Energy_Capacity_dict, index=["Thermal Storage Energy Capacity MWht"])
# r_TES_Discharge_Capacity_df = pd.DataFrame(r_TES_Discharge_Capacity_dict, index=["Thermal Storage Discharge Capacity MWt"])
# r_TES_duration_df = pd.DataFrame(r_TES_duration, index=["Thermal Storage Duration hours"])
# r_TES_on_cost_df = pd.DataFrame(r_TES_on_cost, index=["Thermal Storage Overnight Cost $"])
# r_TES_ann_cost_df = pd.DataFrame(r_TES_ann_cost, index=["Thermal Storage Annualized Cost $/year"])
# r_TES_total_ann_cost_df = pd.DataFrame(r_TES_total_ann_cost, index=["Thermal Storage Total Annualized Cost $"])

a_cTotal_df = pd.DataFrame(a_cTotal_dict, index=["Total objective cost"])
a_cFix_df = pd.DataFrame(a_cFix_dict, index=["Fixed objective costs"])
a_cVar_df = pd.DataFrame(a_cVar_dict, index=["Variable objective costs"])
a_cNSE_df = pd.DataFrame(a_cNSE_dict, index=["NSE objective costs"])
a_cStart_df = pd.DataFrame(a_cStart_dict, index=["Start-up objective costs"])

# d_cTotal_df = pd.DataFrame(d_cTotal_dict, index=["Total objective cost"])
# d_cFix_df = pd.DataFrame(d_cFix_dict, index=["Fixed objective costs"])
# d_cVar_df = pd.DataFrame(d_cVar_dict, index=["Variable objective costs"])
# d_cNSE_df = pd.DataFrame(d_cNSE_dict, index=["NSE objective costs"])
# d_cStart_df = pd.DataFrame(d_cStart_dict, index=["Start-up objective costs"])

# r_cTotal_df = pd.DataFrame(r_cTotal_dict, index=["Total objective cost"])
# r_cFix_df = pd.DataFrame(r_cFix_dict, index=["Fixed objective costs"])
# r_cVar_df = pd.DataFrame(r_cVar_dict, index=["Variable objective costs"])
# r_cNSE_df = pd.DataFrame(r_cNSE_dict, index=["NSE objective costs"])
# r_cStart_df = pd.DataFrame(r_cStart_dict, index=["Start-up objective costs"])

a_FPP_Total_revenue_df = pd.DataFrame(a_FPP_Total_revenue_dict, index=["FPP Total Revenue"])
a_FPP_Energy_revenue_df = pd.DataFrame(a_FPP_Energy_revenue_dict, index=["FPP Energy Revenue"])
a_FPP_Subsidies_revenue_df = pd.DataFrame(a_FPP_Subsidies_revenue_dict, index=["FPP Subsidies Revenue"])

a_FPP_Total_cost_df = pd.DataFrame(a_FPP_Total_cost_dict, index=["FPP Total Cost"])
a_FPP_InvestmentMW_cost_df = pd.DataFrame(a_FPP_InvestmentMW_cost_dict, index=["FPP Investment MW Cost"])
a_FPP_InvestmentMWh_cost_df = pd.DataFrame(a_FPP_InvestmentMWh_cost_dict, index=["FPP Investment MWh Cost"])
a_FPP_FOMMW_cost_df = pd.DataFrame(a_FPP_FOMMW_cost_dict, index=["FPP Fixed O&M MW Cost"])
a_FPP_FOMMWh_cost_df = pd.DataFrame(a_FPP_FOMMWh_cost_dict, index=["FPP Fixed O&M MWh Cost"])
a_FPP_VOMout_cost_df = pd.DataFrame(a_FPP_VOMout_cost_dict, index=["FPP Variable O&M Out Cost"])
a_FPP_Fuel_cost_df = pd.DataFrame(a_FPP_Fuel_cost_dict, index=["FPP Fuel Cost"])
a_FPP_VOMin_cost_df = pd.DataFrame(a_FPP_VOMin_cost_dict, index=["FPP Variable O&M In Cost"])
a_FPP_Start_cost_df = pd.DataFrame(a_FPP_Start_cost_dict, index=["FPP Start Cost"])
a_FPP_Charge_cost_df = pd.DataFrame(a_FPP_Charge_cost_dict, index=["FPP Charge Cost"])
a_FPP_Emissions_cost_df = pd.DataFrame(a_FPP_Emissions_cost_dict, index=["FPP Emissions Cost"])

# d_FPP_Total_revenue_df = pd.DataFrame(d_FPP_Total_revenue_dict, index=["FPP Total Revenue"])
# d_FPP_Energy_revenue_df = pd.DataFrame(d_FPP_Energy_revenue_dict, index=["FPP Energy Revenue"])
# d_FPP_Subsidies_revenue_df = pd.DataFrame(d_FPP_Subsidies_revenue_dict, index=["FPP Subsidies Revenue"])

# d_FPP_Total_cost_df = pd.DataFrame(d_FPP_Total_cost_dict, index=["FPP Total Cost"])
# d_FPP_InvestmentMW_cost_df = pd.DataFrame(d_FPP_InvestmentMW_cost_dict, index=["FPP Investment MW Cost"])
# d_FPP_InvestmentMWh_cost_df = pd.DataFrame(d_FPP_InvestmentMWh_cost_dict, index=["FPP Investment MWh Cost"])
# d_FPP_FOMMW_cost_df = pd.DataFrame(d_FPP_FOMMW_cost_dict, index=["FPP Fixed O&M MW Cost"])
# d_FPP_FOMMWh_cost_df = pd.DataFrame(d_FPP_FOMMWh_cost_dict, index=["FPP Fixed O&M MWh Cost"])
# d_FPP_VOMout_cost_df = pd.DataFrame(d_FPP_VOMout_cost_dict, index=["FPP Variable O&M Out Cost"])
# d_FPP_Fuel_cost_df = pd.DataFrame(d_FPP_Fuel_cost_dict, index=["FPP Fuel Cost"])
# d_FPP_VOMin_cost_df = pd.DataFrame(d_FPP_VOMin_cost_dict, index=["FPP Variable O&M In Cost"])
# d_FPP_Start_cost_df = pd.DataFrame(d_FPP_Start_cost_dict, index=["FPP Start Cost"])
# d_FPP_Charge_cost_df = pd.DataFrame(d_FPP_Charge_cost_dict, index=["FPP Charge Cost"])
# d_FPP_Emissions_cost_df = pd.DataFrame(d_FPP_Emissions_cost_dict, index=["FPP Emissions Cost"])

# r_FPP_Total_revenue_df = pd.DataFrame(r_FPP_Total_revenue_dict, index=["FPP Total Revenue"])
# r_FPP_Energy_revenue_df = pd.DataFrame(r_FPP_Energy_revenue_dict, index=["FPP Energy Revenue"])
# r_FPP_Subsidies_revenue_df = pd.DataFrame(r_FPP_Subsidies_revenue_dict, index=["FPP Subsidies Revenue"])

# r_FPP_Total_cost_df = pd.DataFrame(r_FPP_Total_cost_dict, index=["FPP Total Cost"])
# r_FPP_InvestmentMW_cost_df = pd.DataFrame(r_FPP_InvestmentMW_cost_dict, index=["FPP Investment MW Cost"])
# r_FPP_InvestmentMWh_cost_df = pd.DataFrame(r_FPP_InvestmentMWh_cost_dict, index=["FPP Investment MWh Cost"])
# r_FPP_FOMMW_cost_df = pd.DataFrame(r_FPP_FOMMW_cost_dict, index=["FPP Fixed O&M MW Cost"])
# r_FPP_FOMMWh_cost_df = pd.DataFrame(r_FPP_FOMMWh_cost_dict, index=["FPP Fixed O&M MWh Cost"])
# r_FPP_VOMout_cost_df = pd.DataFrame(r_FPP_VOMout_cost_dict, index=["FPP Variable O&M Out Cost"])
# r_FPP_Fuel_cost_df = pd.DataFrame(r_FPP_Fuel_cost_dict, index=["FPP Fuel Cost"])
# r_FPP_VOMin_cost_df = pd.DataFrame(r_FPP_VOMin_cost_dict, index=["FPP Variable O&M In Cost"])
# r_FPP_Start_cost_df = pd.DataFrame(r_FPP_Start_cost_dict, index=["FPP Start Cost"])
# r_FPP_Charge_cost_df = pd.DataFrame(r_FPP_Charge_cost_dict, index=["FPP Charge Cost"])
# r_FPP_Emissions_cost_df = pd.DataFrame(r_FPP_Emissions_cost_dict, index=["FPP Emissions Cost"])


empty_df = pd.DataFrame(data=empty_df_dict, index=[""])
# FPP_cost_df = pd.DataFrame(data=FPP_cost_dict, index=["FPP Capital Cost ($/kW_e)"]).astype(float)
# FPP_cCapacity_df = pd.DataFrame(data=FPP_cCapacity_dict, index=["FPP Capacity Constraint"])
# Carbon_constraint_df = pd.DataFrame(data=Carbon_constraint_dict, index=["Carbon Constraint (gCO2eq/kWh_e)"]).astype(float)

# TES_duration_cdf = pd.DataFrame(data=TES_Duration_Capacity_cdict, index=["TES Duration constraint (MWh_t)"])
# TES_discharge_cdf = pd.DataFrame(data=TES_Discharge_Cap_cdict, index=["TES Discharge Capacity constraint (MW_t)"])

TES_max_charge_rate_df = pd.DataFrame(data=TES_max_charge_rate_dict, index=["TES max charge rate (\% charge capacity)"])
TES_max_discharge_rate_df = pd.DataFrame(data=TES_max_discharge_rate_dict, index=["TES max discharge rate (\% discharge capacity)"])
TES_min_net_discharge_rate_df = pd.DataFrame(data=TES_min_net_discharge_rate_dict, index=["TES max net charge rate (\% charge capacity)"])
TES_max_net_discharge_rate_df = pd.DataFrame(data=TES_max_net_discharge_rate_dict, index=["TES max net discharge rate (\% discharge capacity)"])
TES_suspected_attractor_df = pd.DataFrame(data=TES_suspected_attractor_dict, index=["TES suspected attractor (\% SoC)"])

#### Concatenation

In [8]:
absolut_df = pd.concat([
    empty_df,
    a_scenario_years_df,
    a_scenario_yearcount_df,
    a_TES_energy_ccost_ass_df,
    empty_df,
    # FPP_cost_df,
    # FPP_cCapacity_df,
    # Carbon_constraint_df,
    # TES_duration_cdf,
    # TES_discharge_cdf,
    # empty_df,
    TES_max_charge_rate_df,
    TES_max_discharge_rate_df,
    TES_min_net_discharge_rate_df,
    TES_max_net_discharge_rate_df,
    TES_suspected_attractor_df,
    empty_df,
    a_Reactor_Thermal_Capacity_df,
    a_Turbine_Gross_Capacity_df,
    a_Turbine_Net_Capacity_df,
    a_Num_Reactors_df,
    empty_df,
    a_TES_Energy_Capacity_df,
    a_TES_Discharge_Capacity_df,
    a_TES_Charge_Capacity_df,
    a_TES_relDisCha_Capacity_df,
    a_TES_duration_on_df,
    a_TES_duration_df,
    a_TES_on_cost_df,
    a_TES_ann_cost_df,
    a_TES_total_ann_cost_df,
    empty_df,
    a_cTotal_df,
    a_cFix_df,
    a_cVar_df,
    a_cNSE_df,
    a_cStart_df,
    empty_df,
    a_FPP_Total_revenue_df,
    a_FPP_Energy_revenue_df,
    a_FPP_Subsidies_revenue_df,
    empty_df,
    a_FPP_Total_cost_df,
    a_FPP_InvestmentMW_cost_df,
    a_FPP_InvestmentMWh_cost_df,
    a_FPP_FOMMW_cost_df,
    a_FPP_FOMMWh_cost_df,
    a_FPP_VOMout_cost_df,
    a_FPP_Fuel_cost_df,
    a_FPP_VOMin_cost_df,
    a_FPP_Start_cost_df,
    a_FPP_Charge_cost_df,
    a_FPP_Emissions_cost_df
], axis=0)

# delta_df = pd.concat([
#     empty_df,
#     FPP_cost_df,
#     FPP_cCapacity_df,
#     Carbon_constraint_df,
#     TES_duration_cdf,
#     TES_discharge_cdf,
#     empty_df,
#     TES_max_charge_rate_df,
#     TES_max_discharge_rate_df,
#     TES_min_net_discharge_rate_df,
#     TES_max_net_discharge_rate_df,
#     TES_suspected_attractor_df,
#     empty_df,
#     d_Reactor_Thermal_Capacity_df,
#     d_Turbine_Gross_Capacity_df,
#     d_Turbine_Net_Capacity_df,
#     d_Num_Reactors_df,
#     empty_df,
#     d_TES_Energy_Capacity_df,
#     d_TES_Discharge_Capacity_df,
#     d_TES_duration_df,
#     d_TES_on_cost_df,
#     d_TES_ann_cost_df,
#     d_TES_total_ann_cost_df,
#     empty_df,
#     d_cTotal_df,
#     d_cFix_df,
#     d_cVar_df,
#     d_cNSE_df,
#     d_cStart_df,
#     empty_df,
#     d_FPP_Total_revenue_df,
#     d_FPP_Energy_revenue_df,
#     d_FPP_Subsidies_revenue_df,
#     empty_df,
#     d_FPP_Total_cost_df,
#     d_FPP_InvestmentMW_cost_df,
#     d_FPP_InvestmentMWh_cost_df,
#     d_FPP_FOMMW_cost_df,
#     d_FPP_FOMMWh_cost_df,
#     d_FPP_VOMout_cost_df,
#     d_FPP_Fuel_cost_df,
#     d_FPP_VOMin_cost_df,
#     d_FPP_Start_cost_df,
#     d_FPP_Charge_cost_df,
#     d_FPP_Emissions_cost_df
# ], axis=0)

# relative_df = pd.concat([
#     empty_df,
#     FPP_cost_df,
#     FPP_cCapacity_df,
#     Carbon_constraint_df,
#     TES_duration_cdf,
#     TES_discharge_cdf,
#     empty_df,
#     TES_max_charge_rate_df,
#     TES_max_discharge_rate_df,
#     TES_min_net_discharge_rate_df,
#     TES_max_net_discharge_rate_df,
#     TES_suspected_attractor_df,
#     empty_df,
#     r_Reactor_Thermal_Capacity_df,
#     r_Turbine_Gross_Capacity_df,
#     r_Turbine_Net_Capacity_df,
#     r_Num_Reactors_df,
#     empty_df,
#     r_TES_Energy_Capacity_df,
#     r_TES_Discharge_Capacity_df,
#     r_TES_duration_df,
#     r_TES_on_cost_df,
#     r_TES_ann_cost_df,
#     r_TES_total_ann_cost_df,
#     empty_df,
#     r_cTotal_df,
#     r_cFix_df,
#     r_cVar_df,
#     r_cNSE_df,
#     r_cStart_df,
#     empty_df,
#     r_FPP_Total_revenue_df,
#     r_FPP_Energy_revenue_df,
#     r_FPP_Subsidies_revenue_df,
#     empty_df,
#     r_FPP_Total_cost_df,
#     r_FPP_InvestmentMW_cost_df,
#     r_FPP_InvestmentMWh_cost_df,
#     r_FPP_FOMMW_cost_df,
#     r_FPP_FOMMWh_cost_df,
#     r_FPP_VOMout_cost_df,
#     r_FPP_Fuel_cost_df,
#     r_FPP_VOMin_cost_df,
#     r_FPP_Start_cost_df,
#     r_FPP_Charge_cost_df,
#     r_FPP_Emissions_cost_df
# ], axis=0)

# print(absolut_df.columns)
# print(scenario_names_list)

absolut_df.columns = scenario_names_list
# delta_df.columns = scenario_names_list
# relative_df.columns = scenario_names_list

### Excel output

In [9]:
Filename = f"GenX_simulations_comparator.xlsx"
print(f"Writing in: ",Filename)

with pd.ExcelWriter(Filename) as writer:
    absolut_df.to_excel(writer, sheet_name="Absolute values", startrow=1, header=False)
    # delta_df.to_excel(writer, sheet_name="Delta values", startrow=1, header=False)
    # relative_df.to_excel(writer, sheet_name="Relative values", startrow=1, header=False)

    workbook = writer.book

    format_centered_wrap = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'text_wrap': True}) 

    format_centered = workbook.add_format({'align': 'center', 'valign': 'vcenter'})
    format_bold_centered = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'bottom': 2})
    format_USD = workbook.add_format({'num_format': 44})
    format_gCO2eq_per_kWh = workbook.add_format({'num_format': '0.00 "gCO2eq/kWh_e"', 'align': 'center', 'valign': 'vcenter'})

    format_milsep_2dec = workbook.add_format({'num_format': '#,##0.00', 'align': 'right', 'valign': 'vcenter'})
    format_MWt = workbook.add_format({'num_format': '#,##0.00" MW_t"', 'align': 'right', 'valign': 'vcenter'})
    format_MWe = workbook.add_format({'num_format': '#,##0.00" MW_e"', 'align': 'right', 'valign': 'vcenter'})

    format_TES_energy_capacity = workbook.add_format({'num_format': '#,##0.00" MWh_t"', 'align': 'right', 'valign': 'vcenter', 'bold': True})
    format_TES_power_capacity = workbook.add_format({'num_format': '#,##0.00" MW_t"', 'align': 'right', 'valign': 'vcenter', 'bold': True})
    format_TES_hours = workbook.add_format({'num_format': '#,##0.00" hours"', 'align': 'right', 'valign': 'vcenter'})

    format_sci2 = workbook.add_format({'num_format': '0.00E+00', 'align': 'right', 'valign': 'vcenter'})
    format_sci5 = workbook.add_format({'num_format': '0.00000E+00', 'align': 'right', 'valign': 'vcenter'})
    format_sci6 = workbook.add_format({'num_format': '0.000000E+00', 'align': 'right', 'valign': 'vcenter'})
    format_sci7 = workbook.add_format({'num_format': '0.0000000E+00', 'align': 'right', 'valign': 'vcenter'})

    format_percent = workbook.add_format({'num_format': '#,##0.00%', 'align': 'right', 'valign': 'vcenter'})
    format_percent_bold_color = workbook.add_format({'bold': True, "bg_color": Color("Red"), 'num_format': '#,##0.00%', 'align': 'right', 'valign': 'vcenter'})

    column_width = 20

    header_row = 0

    zero_block_first_row = 2
    len_zero_block = 3

    first_block_first_row = zero_block_first_row + len_zero_block + 1
    len_first_block = -1

    second_block_first_row = first_block_first_row + len_first_block + 1
    len_second_block = 5

    third_block_first_row = second_block_first_row + len_second_block + 1
    len_third_block = 4

    fourth_block_first_row = third_block_first_row + len_third_block + 1
    len_fourth_block = 9

    fifth_block_first_row = fourth_block_first_row + len_fourth_block + 1
    len_fifth_block = 5

    sixth_block_first_row = fifth_block_first_row + len_fifth_block + 1
    len_sixth_block = 3

    seventh_block_first_row = sixth_block_first_row + len_sixth_block + 1
    len_seventh_block = 11
    
    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]

        worksheet.write(header_row,0, "Metrics ↓ / Scenario names →")
        worksheet.write(header_row,1, "Baseline scenario")
        
        for i in range(1, len(scenario_names_list)+1):
            writer.sheets[sheet_name].set_column(header_row, i, column_width)  # Adjust column width
            worksheet.write(header_row,i, f"{scenario_names_list[i-1]}")
        # end for

        writer.sheets[sheet_name].freeze_panes(1, 1)

        index_width = 42
        writer.sheets[sheet_name].set_column(header_row, 0, index_width)  # Adjust first column width

        writer.sheets[sheet_name].set_row(header_row, 20, format_bold_centered)  # Format header row

        writer.sheets[sheet_name].set_row(zero_block_first_row, None, format_centered_wrap)
        writer.sheets[sheet_name].set_row(zero_block_first_row + 1, None, format_centered)
        writer.sheets[sheet_name].set_row(zero_block_first_row + 2, None, format_USD)

        writer.sheets[sheet_name].set_row(first_block_first_row, None, format_USD)
        writer.sheets[sheet_name].set_row(first_block_first_row + 1, None, format_centered)
        writer.sheets[sheet_name].set_row(first_block_first_row + 2, None, format_gCO2eq_per_kWh)
        writer.sheets[sheet_name].set_row(first_block_first_row + 3, None, format_centered)
        writer.sheets[sheet_name].set_row(first_block_first_row + 4, None, format_centered)
        writer.sheets[sheet_name].set_row(second_block_first_row, None, format_percent)
        writer.sheets[sheet_name].set_row(second_block_first_row + 1, None, format_percent)
        writer.sheets[sheet_name].set_row(second_block_first_row + 2, None, format_percent)
        writer.sheets[sheet_name].set_row(second_block_first_row + 3, None, format_percent)
        writer.sheets[sheet_name].set_row(second_block_first_row + 4, None, format_percent)

        # writer.sheets[sheet_name].conditional_format(second_block_first_row, 2, second_block_first_row + 1, len(scenario_names_list), {'type':     'cell',
        #                                                                                                                                 'criteria': '<',
        #                                                                                                                                 'value':    .95,
        #                                                                                                                                 'format':   format_percent_bold_color})
        # writer.sheets[sheet_name].conditional_format(second_block_first_row + 2, 2, second_block_first_row + 2, len(scenario_names_list), {'type':     'cell',
        #                                                                                                                                 'criteria': '>',
        #                                                                                                                                 'value':    -.95,
        #                                                                                                                                 'format':   format_percent_bold_color})
        # writer.sheets[sheet_name].conditional_format(second_block_first_row + 3, 2, second_block_first_row + 3, len(scenario_names_list), {'type':     'cell',
        #                                                                                                                                 'criteria': '<',
        #                                                                                                                                 'value':    .95,
        #                                                                                                                                 'format':   format_percent_bold_color})
        # writer.sheets[sheet_name].conditional_format(second_block_first_row + 4, 2, second_block_first_row + 4, len(scenario_names_list), {'type':     'cell',
        #                                                                                                                                 'criteria': '>',
        #                                                                                                                                 'value':    .1,
        #                                                                                                                                 'format':   format_percent_bold_color})
    # end for

    # for sheet_name in ['Absolute values', 'Delta values']:
    for sheet_name in ['Absolute values']:
        writer.sheets[sheet_name].set_row(third_block_first_row, None, format_MWt)
        writer.sheets[sheet_name].set_row(third_block_first_row + 1, None, format_MWe)
        writer.sheets[sheet_name].set_row(third_block_first_row + 2, None, format_MWe)
        writer.sheets[sheet_name].set_row(third_block_first_row + 3, None, format_milsep_2dec)

        writer.sheets[sheet_name].set_row(fourth_block_first_row, None, format_TES_energy_capacity)
        writer.sheets[sheet_name].set_row(fourth_block_first_row + 1, None, format_TES_power_capacity)
        writer.sheets[sheet_name].set_row(fourth_block_first_row + 2, None, format_TES_power_capacity)
        writer.sheets[sheet_name].set_row(fourth_block_first_row + 3, None, format_percent)
        writer.sheets[sheet_name].set_row(fourth_block_first_row + 4, None, format_TES_hours)
        writer.sheets[sheet_name].set_row(fourth_block_first_row + 5, None, format_TES_hours)

        for i in range(fourth_block_first_row + 6, fourth_block_first_row + 6 + 3 + 1):
            writer.sheets[sheet_name].set_row(i, None, format_USD)

        for i in range(fifth_block_first_row, fifth_block_first_row + len_fifth_block):
            writer.sheets[sheet_name].set_row(i, None, format_sci7)
        # end for
        
        for i in range(sixth_block_first_row, sixth_block_first_row + len_sixth_block):
            writer.sheets[sheet_name].set_row(i, None, format_USD)
        # end for

        for i in range(seventh_block_first_row, seventh_block_first_row + len_seventh_block):
            writer.sheets[sheet_name].set_row(i, None, format_USD)
        # end for
    # end for

    # for i in range(second_block_first_row, second_block_first_row + len_second_block + len_third_block + len_fourth_block + len_fifth_block + len_sixth_block + len_seventh_block):
    #     writer.sheets['Relative values'].set_row(i, None, format_sci2)
    # # end for        
# end with

Writing in:  GenX_simulations_comparator.xlsx
